КОД разработан Артёмом Сикачёвым.

Вычисление TF и IDF в изначальном словаре. Затем после стемминга и после лемматизации.

In [7]:
import nltk
from collections import Counter
from nltk.stem import SnowballStemmer
import math

# Скачиваем нужные данные (только один раз)
nltk.download('punkt', quiet=True)

# 4 предложения
texts = [
    "Азбука, Азбукой, Азбуками, Библиотека, Библиотекой, Библиотекой, Студент, Студенты",
    "Азбуке, Азбуку, Азбук, Библиотеке, Библиотеку, Библиотек, Студента, Студентам",
    "Азбуки, Азбукою, Азбуках, Библиотеки, Библиотекою, Библиотеках, Студентом, Студентами",
    "Азбуке, Азбукой, Азбуками, Библиотеки, Библиотекой, Библиотеками, Студенте, Студентах"
]

def tokenize_and_clean(raw_text):
    return [word.strip().lower() for word in raw_text.split(',')]

# Стеммер для русского языка
stemmer = SnowballStemmer("russian")

# Функция для простой лемматизации (замена окончаний)
def simple_lemmatize(word):
    if word.startswith('азбук'):
        return 'азбука'
    if word.startswith('библиотек'):
        return 'библиотека'
    if word.startswith('студент'):
        return 'студент'
    return word

# Функция для вычисления TF (частота слова в документе / общее число слов в документе)
def compute_tf(word_counts, total_words):
    tf = {}
    for word, count in word_counts.items():
        tf[word] = count / total_words
    return tf

# Функция для вычисления IDF (log(общее число документов / число документов с этим словом))
def compute_idf(word_doc_counts, total_docs):
    idf = {}
    for word, doc_count in word_doc_counts.items():
        idf[word] = math.log(total_docs / doc_count)
    return idf

# Обрабатываем каждый вариант отдельно
def process_variant(variant_name, process_func):
    print(f"\n{'='*60}")
    print(f"{variant_name}")
    print('='*60)
    
    # Токенизируем каждое предложение отдельно
    tokenized_docs = []
    for text in texts:
        words_raw = tokenize_and_clean(text)
        processed_words = [process_func(w) for w in words_raw]
        tokenized_docs.append(processed_words)
    
    # Собираем все слова для общего мешка слов
    all_words = []
    for doc in tokenized_docs:
        all_words.extend(doc)
    
    # 1. Мешок слов (общие частоты)
    freq = Counter(all_words)
    print(f"\n1) Мешок слов (все предложения вместе):")
    print(f"   Уникальных слов: {len(set(all_words))}")
    print(f"   Частоты: {dict(freq)}")
    
    # 2. TF для каждого документа
    print(f"\n2) TF (Term Frequency) для каждого предложения:")
    for i, doc in enumerate(tokenized_docs, 1):
        doc_freq = Counter(doc)
        tf = compute_tf(doc_freq, len(doc))
        print(f"   Предложение {i}: {tf}")
    
    # 3. IDF
    print(f"\n3) IDF (Inverse Document Frequency):")
    # Считаем, в скольких документах встречается каждое слово
    word_doc_counts = {}
    for doc in tokenized_docs:
        unique_words_in_doc = set(doc)
        for word in unique_words_in_doc:
            word_doc_counts[word] = word_doc_counts.get(word, 0) + 1
    
    idf = compute_idf(word_doc_counts, len(texts))
    print(f"   {idf}")
    
    return freq, tokenized_docs

# Запускаем обработку для трёх вариантов

# 1) Без обработки
process_variant("ВАРИАНТ 1: БЕЗ ОБРАБОТКИ", lambda w: w.strip().lower())

# 2) После стемминга
process_variant("ВАРИАНТ 2: ПОСЛЕ СТЕММИНГА (Snowball)", lambda w: stemmer.stem(w.strip().lower()))

# 3) После лемматизации
process_variant("ВАРИАНТ 3: ПОСЛЕ ЛЕММАТИЗАЦИИ (простая замена)", lambda w: simple_lemmatize(w.strip().lower()))


ВАРИАНТ 1: БЕЗ ОБРАБОТКИ

1) Мешок слов (все предложения вместе):
   Уникальных слов: 26
   Частоты: {'азбука': 1, 'азбукой': 2, 'азбуками': 2, 'библиотека': 1, 'библиотекой': 3, 'студент': 1, 'студенты': 1, 'азбуке': 2, 'азбуку': 1, 'азбук': 1, 'библиотеке': 1, 'библиотеку': 1, 'библиотек': 1, 'студента': 1, 'студентам': 1, 'азбуки': 1, 'азбукою': 1, 'азбуках': 1, 'библиотеки': 2, 'библиотекою': 1, 'библиотеках': 1, 'студентом': 1, 'студентами': 1, 'библиотеками': 1, 'студенте': 1, 'студентах': 1}

2) TF (Term Frequency) для каждого предложения:
   Предложение 1: {'азбука': 0.125, 'азбукой': 0.125, 'азбуками': 0.125, 'библиотека': 0.125, 'библиотекой': 0.25, 'студент': 0.125, 'студенты': 0.125}
   Предложение 2: {'азбуке': 0.125, 'азбуку': 0.125, 'азбук': 0.125, 'библиотеке': 0.125, 'библиотеку': 0.125, 'библиотек': 0.125, 'студента': 0.125, 'студентам': 0.125}
   Предложение 3: {'азбуки': 0.125, 'азбукою': 0.125, 'азбуках': 0.125, 'библиотеки': 0.125, 'библиотекою': 0.125, 'библиоте

(Counter({'азбука': 12, 'библиотека': 12, 'студент': 8}),
 [['азбука',
   'азбука',
   'азбука',
   'библиотека',
   'библиотека',
   'библиотека',
   'студент',
   'студент'],
  ['азбука',
   'азбука',
   'азбука',
   'библиотека',
   'библиотека',
   'библиотека',
   'студент',
   'студент'],
  ['азбука',
   'азбука',
   'азбука',
   'библиотека',
   'библиотека',
   'библиотека',
   'студент',
   'студент'],
  ['азбука',
   'азбука',
   'азбука',
   'библиотека',
   'библиотека',
   'библиотека',
   'студент',
   'студент']])